# Séance 2 — Pandas : structures et exploration

**Durée pratique : 3 h 30** &nbsp;·&nbsp; Ateliers 2.1 à 2.3

## Ce que vous saurez faire à la fin

- charger une même source depuis trois formats et réconcilier les types obtenus ;
- sélectionner et filtrer sans déclencher de `SettingWithCopyWarning` ;
- produire automatiquement un rapport d'exploration sur un jeu de données inconnu.

Le jeu de travail est `ventes_brutes.csv` : environ 17 600 lignes de commandes,
volontairement dégradées. Vous allez le retrouver à chaque séance jusqu'à la fin du module.

> **Convention de nommage du module.** Le code est écrit en anglais et suit la PEP 8 :
> fonctions et variables en `snake_case`, constantes en `MAJUSCULES`. Les **noms de colonnes**
> restent en français parce qu'ils viennent de la source : renommer les colonnes d'un fichier
> d'entrée est une transformation comme une autre, elle se décide et se documente, elle ne se
> fait pas par réflexe. Vous rencontrerez cette situation partout en entreprise.

In [5]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_DIR = ROOT / 'data' / 'raw'
PROCESSED_DIR = ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)

print('Données disponibles :')
for path in sorted(RAW_DIR.glob('*')):
    print(' ', path.name)


# Parquet conserve les types (dates, entiers, catégories) là où le CSV les perd :
# c'est le format à privilégier entre deux étapes d'un pipeline. Repli automatique
# sur le CSV si pyarrow n'est pas installé.
def save_dataset(df, name):
    try:
        path = PROCESSED_DIR / f'{name}.parquet'
        df.to_parquet(path, index=False)
    except ImportError:
        path = PROCESSED_DIR / f'{name}.csv'
        df.to_csv(path, index=False)
        print('(pyarrow absent : repli sur le CSV)')
    print('écrit :', path.name, df.shape)
    return path


def load_dataset(name):
    parquet_path = PROCESSED_DIR / f'{name}.parquet'
    csv_path = PROCESSED_DIR / f'{name}.csv'
    if parquet_path.exists():
        return pd.read_parquet(parquet_path)
    if csv_path.exists():
        return pd.read_csv(csv_path)
    raise FileNotFoundError(f'{name} introuvable : exécutez le notebook précédent')


def dataset_exists(name):
    return ((PROCESSED_DIR / f'{name}.parquet').exists()
            or (PROCESSED_DIR / f'{name}.csv').exists())

ImportError: DLL load failed while importing writers: Une stratégie de contrôle d’application a bloqué ce fichier.

---
## Atelier 2.1 — Lire une source, vraiment (50 min)

### Partie guidée : la lecture naïve et ce qu'elle cache

In [ ]:
sales = pd.read_csv(RAW_DIR / 'ventes_brutes.csv')

print('Dimensions :', sales.shape)
sales.head()

In [ ]:
sales.dtypes

Regardez `prix_unitaire`. Le type est `object`, autrement dit du texte, alors qu'il s'agit
d'un montant. Cherchons pourquoi.

In [ ]:
# Isoler les valeurs qui ne se convertissent pas en nombre
as_number = pd.to_numeric(sales['prix_unitaire'], errors='coerce')
unparsable = sales.loc[as_number.isna(), 'prix_unitaire']

print('Valeurs non convertibles :', len(unparsable))
print(unparsable.head(8).tolist())

Une partie des prix a été saisie avec une virgule décimale et un symbole monétaire.
Une lecture qui ignore ce détail produit une colonne texte, et tout calcul en aval échoue
silencieusement ou renvoie une erreur bien plus loin dans le pipeline.

**C'est la règle à retenir : ne jamais faire confiance à l'inférence de types.** Vérifiez
systématiquement `dtypes` après chaque lecture.

In [ ]:
def parse_price(series):
    """Convertit une colonne de prix mixte (nombre ou texte '123,45 EUR') en float."""
    text = series.astype(str).str.replace(' EUR', '', regex=False)
    text = text.str.replace(',', '.', regex=False).str.strip()
    return pd.to_numeric(text, errors='coerce')


unit_price = parse_price(sales['prix_unitaire'])
print('Valeurs encore non convertibles :', unit_price.isna().sum())
print(unit_price.describe().round(2))

### Partie autonome

In [ ]:
# Q1. Charger l'extrait des ventes depuis les trois formats disponibles,
#     puis comparer les dtypes obtenus pour la colonne 'date_commande'.
from pathlib import Path

import pandas as pd

RAW_DIR = Path("data/raw")


sample_csv = pd.read_csv(RAW_DIR /'ventes_brutes.csv')
sample_xlsx = pd.read_excel(RAW_DIR /'ventes_extrait.xlsx')
sample_json = pd.read_json(RAW_DIR /'ventes_extrait.json')

for label, frame in [('csv', sample_csv), ('xlsx', sample_xlsx), ('json', sample_json)]:
    print(f"{label:<6} lignes={len(frame):<6} dtype date={frame['date_commande'].dtype}")


**Question à traiter par écrit dans la cellule suivante.** Les trois formats donnent-ils
le même type pour `date_commande` ? Le même nombre de lignes ? Que se passerait-il si votre
pipeline acceptait indifféremment ces trois sources ?

*Votre réponse :*



In [ ]:
# Q2. Relire ventes_brutes.csv en une seule instruction, en imposant :
#     - id_client, id_produit, id_magasin comme chaînes de caractères ;
#     - la chaîne vide et 'NC' comme valeurs manquantes ;
#     - seulement les colonnes id_commande, date_commande, id_produit, quantite,
#       prix_unitaire, statut.
#     Indice : paramètres dtype, na_values et usecols.

EXPECTED_COLUMNS = ['id_commande', 'date_commande', 'id_produit',
                    'quantite', 'prix_unitaire', 'statut']

sales_subset = pd.read_csv(
    RAW_DIR / 'ventes_brutes.csv',
    dtype={
        'id_client': object,
        'id_produit': object,
        'id_magasin': object
    },
    na_values=['', 'NC'],
    usecols=EXPECTED_COLUMNS
)

assert list(sales_subset.columns) == EXPECTED_COLUMNS
assert sales_subset['id_produit'].dtype == object

print('OK —', sales_subset.shape)

---
## Atelier 2.2 — Sélection et filtrage (60 min)

### Partie guidée : `loc`, `iloc` et le piège de la copie

In [ ]:
sales['prix_unitaire'] = parse_price(sales['prix_unitaire'])
sales['montant'] = sales['quantite'] * sales['prix_unitaire']

# loc travaille sur les ÉTIQUETTES (noms de colonnes, valeurs d'index)
print(sales.loc[0:2, ['id_commande', 'quantite', 'montant']])
print()
# iloc travaille sur les POSITIONS entières
print(sales.iloc[0:2, [0, 5, -1]])

Notez la différence sur les bornes : `loc[0:2]` renvoie **trois** lignes (borne incluse),
`iloc[0:2]` en renvoie **deux** (borne exclue, comme le slicing Python).

In [ ]:
# Filtrage booléen : combiner des conditions avec & et |, chaque condition entre parenthèses
large_orders = sales[(sales['montant'] > 1000) & (sales['statut'] == 'livre')]
print('Commandes livrées de plus de 1000 EUR :', len(large_orders))

# query() est souvent plus lisible quand les conditions s'accumulent
same_result = sales.query("montant > 1000 and statut == 'livre'")
print('Même résultat :', len(same_result) == len(large_orders))

In [ ]:
# LE PIÈGE : modifier un sous-ensemble extrait par filtrage
cancelled = sales[sales['statut'] == 'annule']

# La ligne suivante déclenche un SettingWithCopyWarning : pandas ne sait pas si
# `cancelled` est une vue sur `sales` ou une copie indépendante.
cancelled['montant'] = 0

print('Montant dans sales pour les annulées :',
      sales.loc[sales['statut'] == 'annule', 'montant'].head(3).tolist())
print("-> la modification n'a PAS été propagée : le travail est perdu")

**Les deux écritures correctes**, selon l'intention :

```python
# Intention A : modifier le DataFrame d'origine
sales.loc[sales['statut'] == 'annule', 'montant'] = 0

# Intention B : travailler sur une copie indépendante
cancelled = sales[sales['statut'] == 'annule'].copy()
cancelled['montant'] = 0
```

Le message d'avertissement est le symptôme d'une ambiguïté dans votre code, pas un bruit
à faire taire.

### Partie autonome

In [ ]:
# Q3. Extraire les commandes qui remplissent TOUTES ces conditions :
#     - statut 'livre' ;
#     - quantite comprise entre 1 et 10 inclus ;
#     - montant strictement supérieur à la médiane des montants des commandes livrées ;
#     - canal contenant 'web', quelle que soit la casse.
#     Le résultat doit être une COPIE indépendante.

target_orders = sales[
    (sales['statut'] == 'livre')
    & (sales['quantite'].between(1, 10))
    & (sales['montant'] > sales.loc[sales['statut'] == 'livre', 'montant'].median())
    & (sales['canal'].str.strip().str.lower().eq('web'))
].copy()

assert isinstance(target_orders, pd.DataFrame)
assert target_orders['quantite'].between(1, 10).all()
assert (target_orders['statut'] == 'livre').all()
assert target_orders['canal'].str.strip().str.lower().eq('web').all()

print('OK —', len(target_orders), 'commandes retenues')

In [ ]:
# Q4. Sans utiliser groupby, calculer le montant total des commandes livrées
#     pour chacun des trois canaux (après normalisation de la casse).
#     Un dictionnaire {canal: total} est attendu.

revenue_by_channel = {}

for channel in ['web', 'boutique', 'telephone']:

    filtered_sales = sales.loc[
        (sales['statut'] == 'livre')
        & (sales['canal'].str.strip().str.lower() == channel),
        'montant'
    ]

    revenue_by_channel[channel] = filtered_sales.sum()

assert set(revenue_by_channel) == {'web', 'boutique', 'telephone'}
print('OK')
for channel, total in sorted(revenue_by_channel.items(), key=lambda item: -item[1]):
    print(f'  {channel:<12} {total:>14,.2f} EUR'.replace(',', ' '))

---
## Atelier 2.3 — Un rapport d'exploration réutilisable (100 min)

Face à un jeu de données inconnu, les mêmes questions reviennent toujours. Plutôt que de
les reposer à la main à chaque fois, vous allez écrire une fonction qui y répond.
**Cette fonction vous servira jusqu'à la fin du module, y compris sur votre projet.**

### Partie guidée : les briques

In [ ]:
print('--- shape ---'); print(sales.shape)
print('\n--- info ---'); sales.info(memory_usage='deep')

In [ ]:
print('--- taux de valeurs manquantes ---')
missing_rate = (sales.isna().mean() * 100).round(2).sort_values(ascending=False)
print(missing_rate[missing_rate > 0])

print('\n--- cardinalité des colonnes texte ---')
for column in sales.select_dtypes(include='object').columns:
    print(f'  {column:<18} {sales[column].nunique():>6} modalités')

In [ ]:
print('--- modalités de canal, telles quelles ---')
print(sales['canal'].value_counts())

Neuf modalités pour ce qui devrait en compter trois. La casse et les espaces parasites
créent de faux niveaux. Un `value_counts()` brut est précisément l'outil qui révèle ce
genre de problème : c'est pourquoi il doit figurer dans le rapport automatique.

### Partie autonome : la fonction `profile_dataframe`

In [ ]:
def profile_dataframe(df, name='jeu de données', max_cardinality=25):
    """Affiche un rapport d'exploration standard.

    Doit produire, dans cet ordre :
      1. le nom, les dimensions et l'empreinte mémoire ;
      2. le nombre de lignes strictement dupliquées ;
      3. un tableau par colonne : type, nombre de valeurs manquantes, taux en %,
         nombre de valeurs distinctes ;
      4. les statistiques descriptives des colonnes numériques ;
      5. pour chaque colonne texte de cardinalité inférieure à max_cardinality,
         la répartition des modalités.

    Ne renvoie rien : la fonction affiche.
    """
    # TODO


    # 1. Nom, dimensions, mémoire
    print(f"=== Profil du {name} ===")
    print(f"Dimensions : {df.shape[0]} lignes × {df.shape[1]} colonnes")
    print(f"Empreinte mémoire : {df.memory_usage(deep=True).sum() / 1024:.1f} Ko\n")

    # 2. Lignes dupliquées
    duplicates = df.duplicated().sum()
    print(f"Lignes strictement dupliquées : {duplicates}\n")

    # 3. Tableau par colonne
    print("=== Colonnes ===")
    summary = []
    for col in df.columns:
        missing = df[col].isna().sum()
        missing_pct = 100 * missing / len(df)
        distinct = df[col].nunique(dropna=True)
        summary.append([col, df[col].dtype, missing, f"{missing_pct:.2f}%", distinct])

    print(pd.DataFrame(summary, columns=[
        "colonne", "dtype", "manquants", "% manquants", "distincts"
    ]))
    print()

    # 4. Statistiques descriptives
    print("=== Statistiques descriptives (numériques) ===")
    print(df.describe(include='number'))
    print()

    # 5. Modalités des colonnes texte à faible cardinalité
    print("=== Modalités des colonnes texte ===")
    for col in df.select_dtypes(include=['object', 'string']):
        if df[col].nunique() <= max_cardinality:
            print(f"\nRépartition pour '{col}' :")
            print(df[col].value_counts(dropna=False))


profile_dataframe(sales, name='ventes_brutes')

In [ ]:
# Q5. Appliquer la fonction aux deux autres jeux de données et vérifier qu'elle
#     se comporte correctement sur des structures différentes.

customers = pd.read_csv(RAW_DIR / 'clients.csv')
sensors = pd.read_csv(RAW_DIR / 'capteurs.csv')

profile_dataframe(customers, name='clients')
profile_dataframe(sensors, name='capteurs')

In [ ]:
# Q6. Déplacer la fonction dans src/exploration.py, puis l'importer ici.
#     Le notebook doit rester lisible : le code réutilisable vit dans src/.

import sys
sys.path.insert(0, str(ROOT / 'src'))

# from exploration import profile_dataframe   # décommentez une fois le fichier créé

---
## Ce que le rapport révèle déjà

Rédigez ici, en cinq à dix lignes, la liste des anomalies que votre rapport a mises au jour
sur `ventes_brutes`. Ce texte est le point de départ de la séance 3 et le premier élément
de votre note méthodologique de projet.

*Vos observations :*

1. 
2. 
3. 

---
## Livrable de la séance

- `src/exploration.py` contenant la fonction `profile_dataframe`, importable ;
- ce notebook exécuté, avec les six questions complétées ;
- la liste écrite des anomalies constatées.